# Scopus publication-level dataset per author (Notebook)

This notebook builds a **publication-level** dataset for each professor/author using the **Elsevier Scopus Search API** (via `requests`, not `pybliometrics`), which avoids the `service level` issues you hit.

## Output
A tidy table with **one row per publication per author**, including:
- author metadata (author_id, professor name, optional internal id)
- publication metadata (title, year, source/journal, ISSN/eISSN, DOI, subtype, etc.)
- impact (citedby-count)
- collaboration signals (coauthor count; country set if available from search payload)
- raw JSON blobs for traceability (optional)

> Q1/Q2 requires an external journal-quartile lookup (SJR/CiteScore). A placeholder section is included.


In [34]:
# If needed, install dependencies:
# !pip install pandas requests openpyxl pyarrow

import os
import time
import random
import requests
import pandas as pd
from datetime import datetime


## 1) Configuration

In [35]:
# Put your key in an environment variable (recommended):
os.environ["ELSEVIER_API_KEY"] = "***REMOVED***"
API_KEY = os.getenv("ELSEVIER_API_KEY")
assert API_KEY, "Missing ELSEVIER_API_KEY env var"

BASE_URL = "https://api.elsevier.com/content/search/scopus"

# Default time window (edit as needed)
START_YEAR = 2019
END_YEAR   = 2025

# Throttling (edit as needed)
SLEEP_MIN = 0.2
SLEEP_MAX = 0.6

# Safety caps (edit as needed)
PER_PAGE = 25           # Scopus Search API typical max is 25
MAX_RECORDS_PER_AUTHOR = 5000  # cap to avoid runaway authors

# Output folder
OUT_DIR = "scopus_out"
DATA_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)

print("Config loaded:", datetime.now().isoformat())


Config loaded: 2026-02-05T14:49:36.108197


## 2) Define the professor/author roster
Use Scopus Author IDs whenever possible (recommended).

In [36]:
# Minimal roster example:
# You can also load from an Excel/CSV roster.

authors = pd.DataFrame([
    {"prof_id": "TEC-001", "prof_name": "Molina-Perez, Edmundo", "author_id": "56709531800"},
    {"prof_id": "TEC-002", "prof_name": "Stein, Ernesto H.", "author_id": "7202194915"},
    {"prof_id": "TEC-003", "prof_name": "Bernal-Serrano, Daniel", "author_id": "57215937382"},
    {"prof_id": "TEC-004", "prof_name": "Aguilar-Gomez, Sandra", "author_id": "57205638710"},
    {"prof_id": "TEC-005", "prof_name": "Campos-Rivera, Paola Abril", "author_id": "57190304109"},
    {"prof_id": "TEC-006", "prof_name": "Garcia, Magdalena", "author_id": "60054037400"},
    {"prof_id": "TEC-007", "prof_name": "Duran-Fernandez, Roberto", "author_id": "55349505800"},
    {"prof_id": "TEC-008", "prof_name": "Santos, Miguel Angel", "author_id": "57015987800"},
    {"prof_id": "TEC-009", "prof_name": "Popper, Steven W.", "author_id": "7003635400"},
    {"prof_id": "TEC-010", "prof_name": "Peraza-Mues, Gonzalo G.", "author_id": "54997963100"},
    {"prof_id": "TEC-011", "prof_name": "Sobrino, Fernanda", "author_id": "59573511100"},
    {"prof_id": "TEC-012", "prof_name": "Benavides-Rincon, Guillermina", "author_id": "57216153915"},
    {"prof_id": "TEC-013", "prof_name": "Morales-Arilla, Jose", "author_id": "57282638100"},
    {"prof_id": "TEC-014", "prof_name": "Contreras-Loya, David", "author_id": "55971186800"},
    {"prof_id": "TEC-015", "prof_name": "Ponce-Lopez, Roberto", "author_id": "57205704459"},
    {"prof_id": "TEC-016", "prof_name": "Gomez-Zaldivar, Fernando", "author_id": "57219596780"},
    {"prof_id": "TEC-017", "prof_name": "Syme, James", "author_id": "57216353171"},
    {"prof_id": "TEC-018", "prof_name": "Diaz-Dominguez, Alejandro", "author_id": "55581628300"},
    {"prof_id": "TEC-019", "prof_name": "Silverio-Murillo, Adan", "author_id": "57212534001"},
    {"prof_id": "TEC-020", "prof_name": "Villarreal, Héctor J.", "author_id": "36855894400"},
    {"prof_id": "TEC-021", "prof_name": "De Unánue, Adolfo", "author_id": "56013629800"},
])

authors


,prof_id,prof_name,author_id
0,TEC-001,"Molina-Perez, Edmundo",56709531800
1,TEC-002,"Stein, Ernesto H.",7202194915
2,TEC-003,"Bernal-Serrano, Daniel",57215937382
3,TEC-004,"Aguilar-Gomez, Sandra",57205638710
4,TEC-005,"Campos-Rivera, Paola Abril",57190304109
5,TEC-006,"Garcia, Magdalena",60054037400
6,TEC-007,"Duran-Fernandez, Roberto",55349505800
7,TEC-008,"Santos, Miguel Angel",57015987800
8,TEC-009,"Popper, Steven W.",7003635400
9,TEC-010,"Peraza-Mues, Gonzalo G.",54997963100


## 3) Low-level Scopus Search API helpers

In [37]:
def scopus_search(query: str, start: int = 0, count: int = 25, *, timeout: int = 60):
    headers = {"X-ELS-APIKey": API_KEY, "Accept": "application/json"}
    params = {"query": query, "start": start, "count": count}
    r = requests.get(BASE_URL, headers=headers, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json(), r.headers

def build_author_query(author_id: str, start_year: int, end_year: int) -> str:
    # inclusive years
    return f"AU-ID({author_id}) AND PUBYEAR > {start_year-1} AND PUBYEAR < {end_year+1}"

def sleep_a_bit():
    time.sleep(SLEEP_MIN + random.random() * (SLEEP_MAX - SLEEP_MIN))


## 4) Fetch all publications for one author (paged)

In [38]:
def fetch_author_publications(author_id: str, start_year: int, end_year: int,
                              per_page: int = PER_PAGE, max_records: int = MAX_RECORDS_PER_AUTHOR):
    query = build_author_query(author_id, start_year, end_year)

    # first request to get total
    js, hdr = scopus_search(query, start=0, count=1)
    total = int(js["search-results"]["opensearch:totalResults"])
    capped_total = min(total, max_records)

    entries_all = []
    for start in range(0, capped_total, per_page):
        js, _ = scopus_search(query, start=start, count=per_page)
        entries = js["search-results"].get("entry", [])
        entries_all.extend(entries)
        sleep_a_bit()

    return entries_all, {"total": total, "capped_total": capped_total, "query": query}


## 5) Normalize entries to a publication-level table

In [39]:
def normalize_entries(entries: list) -> pd.DataFrame:
    if not entries:
        return pd.DataFrame()
    df = pd.json_normalize(entries)

    # Common useful fields (existence depends on result)
    wanted = [
        "dc:identifier",           # SCOPUS_ID
        "eid",
        "dc:title",
        "dc:creator",              # usually first author string
        "prism:coverDate",
        "prism:publicationName",
        "prism:issn",
        "prism:eIssn",
        "prism:doi",
        "subtype",
        "subtypeDescription",
        "citedby-count",
        "prism:aggregationType",
        "prism:url",
        "openaccess",
        "openaccessFlag",
        "authkeywords",
        "affiliation",                                         
        "affiliation-name",
        "affiliation-country",                         
        "author-count",
    ]
    keep = [c for c in wanted if c in df.columns]
    df = df[keep].copy()

    # Parse year
    if "prism:coverDate" in df.columns:
        df["year"] = pd.to_datetime(df["prism:coverDate"], errors="coerce").dt.year

    # numeric citations
    if "citedby-count" in df.columns:
        df["citedby-count"] = pd.to_numeric(df["citedby-count"], errors="coerce")

    # Coauthor count (sometimes present)
    if "author-count" in df.columns:
        # author-count sometimes nested or dict; try to extract '@total'
        def _ac(x):
            if isinstance(x, dict):
                return pd.to_numeric(x.get("@total"), errors="coerce")
            return pd.to_numeric(x, errors="coerce")
        df["coauthor_count"] = df["author-count"].apply(_ac)
        df.drop(columns=["author-count"], inplace=True, errors="ignore")

    # Extract affiliation countries if present (for international collab proxy)
    if "affiliation" in df.columns:
        def _countries(aff):
            if not isinstance(aff, list):
                return None
            c = []
            for a in aff:
                if isinstance(a, dict):
                    cc = a.get("affiliation-country")
                    if cc:
                        c.append(cc)
            c = sorted(set(c))
            return "; ".join(c) if c else None

        def _aff_names(aff):
            if not isinstance(aff, list):
                return None
            n = []
            for a in aff:
                if isinstance(a, dict):
                    nm = a.get("affilname")
                    if nm:
                        n.append(nm)
            n = sorted(set(n))
            return "; ".join(n) if n else None

        df["affiliation_countries"] = df["affiliation"].apply(_countries)
        df["affiliation_names"] = df["affiliation"].apply(_aff_names)
        # Keep raw affiliation for traceability if you want; otherwise drop:
        # df.drop(columns=["affiliation"], inplace=True)

    # Derive a clean scopus_id
    if "dc:identifier" in df.columns:
        # format like "SCOPUS_ID:1234567890"
        df["scopus_id"] = df["dc:identifier"].astype(str).str.replace("SCOPUS_ID:", "", regex=False)

    return df


## 6) Build the final dataset (one row per publication per author)

In [40]:
def build_publication_level_dataset(authors_df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    out_frames = []
    logs = []

    for _, row in authors_df.iterrows():
        author_id = str(row["author_id"])
        prof_id = row.get("prof_id", None)
        prof_name = row.get("prof_name", None)

        print(f"Fetching AU-ID({author_id}) {start_year}-{end_year} ...")
        entries, meta = fetch_author_publications(author_id, start_year, end_year)
        logs.append({"author_id": author_id, "prof_id": prof_id, "prof_name": prof_name, **meta})

        df = normalize_entries(entries)
        #df = pd.json_normalize(entries)
        if df.empty:
            continue

        # add author dimensions
        df.insert(0, "author_id", author_id)
        df.insert(1, "prof_id", prof_id)
        df.insert(2, "prof_name", prof_name)

        out_frames.append(df)

    log_df = pd.DataFrame(logs)
    log_path = os.path.join(OUT_DIR, "fetch_log.csv")
    log_df.to_csv(log_path, index=False)
    print("Saved fetch log:", log_path)

    if not out_frames:
        return pd.DataFrame()

    final = pd.concat(out_frames, ignore_index=True)
    return final

pubs = build_publication_level_dataset(authors, START_YEAR, END_YEAR)
pubs.shape, pubs.head(3)


Fetching AU-ID(56709531800) 2019-2025 ...
Fetching AU-ID(7202194915) 2019-2025 ...
Fetching AU-ID(57215937382) 2019-2025 ...
Fetching AU-ID(57205638710) 2019-2025 ...
Fetching AU-ID(57190304109) 2019-2025 ...
Fetching AU-ID(60054037400) 2019-2025 ...
Fetching AU-ID(55349505800) 2019-2025 ...
Fetching AU-ID(57015987800) 2019-2025 ...
Fetching AU-ID(7003635400) 2019-2025 ...
Fetching AU-ID(54997963100) 2019-2025 ...
Fetching AU-ID(59573511100) 2019-2025 ...
Fetching AU-ID(57216153915) 2019-2025 ...
Fetching AU-ID(57282638100) 2019-2025 ...
Fetching AU-ID(55971186800) 2019-2025 ...
Fetching AU-ID(57205704459) 2019-2025 ...
Fetching AU-ID(57219596780) 2019-2025 ...
Fetching AU-ID(57216353171) 2019-2025 ...
Fetching AU-ID(55581628300) 2019-2025 ...
Fetching AU-ID(57212534001) 2019-2025 ...
Fetching AU-ID(36855894400) 2019-2025 ...
Fetching AU-ID(56013629800) 2019-2025 ...
Saved fetch log: scopus_out/fetch_log.csv


((167, 24),
      author_id  prof_id              prof_name           dc:identifier  \
 0  56709531800  TEC-001  Molina-Perez, Edmundo  SCOPUS_ID:105015077202   
 1  56709531800  TEC-001  Molina-Perez, Edmundo  SCOPUS_ID:105013320148   
 2  56709531800  TEC-001  Molina-Perez, Edmundo   SCOPUS_ID:85185156225   
 
                    eid                                           dc:title  \
 0  2-s2.0-105015077202  Groundwater parameters estimation: A hybrid ap...   
 1  2-s2.0-105013320148  Using a robust decision making (RDM) approach ...   
 2   2-s2.0-85185156225  The economics of decarbonizing Costa Rica's ag...   
 
     dc:creator prism:coverDate  \
 0     Tesen K.      2025-12-09   
 1   Poblete D.      2025-01-01   
 2  Banerjee O.      2024-04-01   
 
                                prism:publicationName prism:issn  ...  \
 0  Engineering Applications of Artificial Intelli...   09521976  ...   
 1                 Frontiers in Environmental Science        NaN  ...   
 2         

In [41]:
pubs.iloc[7]

author_id                                                      56709531800
prof_id                                                            TEC-001
prof_name                                            Molina-Perez, Edmundo
dc:identifier                                        SCOPUS_ID:85164252140
eid                                                     2-s2.0-85164252140
dc:title                 Knowledge co-production for decision-making in...
dc:creator                                                   Moallemi E.A.
prism:coverDate                                                 2023-09-01
prism:publicationName                          Global Environmental Change
prism:issn                                                        09593780
prism:eIssn                                                            NaN
prism:doi                                  10.1016/j.gloenvcha.2023.102727
subtype                                                                 ar
subtypeDescription       

In [42]:
pubs.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,citedby-count,prism:aggregationType,prism:url,openaccess,openaccessFlag,affiliation,year,affiliation_countries,affiliation_names,scopus_id
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,0,Journal,https://api.elsevier.com/content/abstract/scop...,0,False,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile; Peru,Pontificia Universidad Católica de Chile; Univ...,105015077202
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,0,Journal,https://api.elsevier.com/content/abstract/scop...,1,True,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile,Pontificia Universidad Católica de Chile; Univ...,105013320148
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,6,Journal,https://api.elsevier.com/content/abstract/scop...,2,False,"[{'@_fa': 'true', 'affilname': 'RMGEO Consulta...",2024,Canada,RMGEO Consultants Inc.,85185156225


## 7) Journal quartiles
Provide a journal lookup table with ISSN/eISSN -> quartile.


In [43]:
# Example expected schema for journal_quartiles.csv:
# issn, eissn, year, quartile  (quartile in {Q1,Q2,Q3,Q4})
# Load and left-join on issn/eissn + year.

ql = pd.read_csv(os.path.join(DATA_DIR, "scimagojr 2024.csv"), sep=";")
ql.head(3)

,Rank,Sourceid,Title,Type,Issn,Publisher,Open Access,Open Access Diamond,SJR,SJR Best Quartile,...,Ref. / Doc.,%Female,Overton,SDG,Country,Region,Publisher.1,Coverage,Categories,Areas
0,1,28773,Ca-A Cancer Journal for Clinicians,journal,"15424863, 00079235",John Wiley and Sons Inc,No,No,"145,004",Q1,...,"62,88","48,21",4,37,United States,Northern America,John Wiley and Sons Inc,1950-2025,Hematology (Q1); Oncology (Q1),Medicine
1,2,19434,MMWR Recommendations and Reports,journal,"10575987, 15458601",Centers for Disease Control and Prevention (CDC),Yes,No,"41,754",Q1,...,"275,33","75,93",1,5,United States,Northern America,Centers for Disease Control and Prevention (CDC),1990-2024,Epidemiology (Q1); Health Information Manageme...,Environmental Science; Health Professions; Med...
2,3,20315,Nature Reviews Molecular Cell Biology,journal,"14710072, 14710080",Nature Research,No,No,"37,353",Q1,...,"92,45","41,22",0,15,United Kingdom,Western Europe,Nature Research,2000-2025,Cell Biology (Q1); Molecular Biology (Q1),"Biochemistry, Genetics and Molecular Biology"


In [44]:
ql

,Rank,Sourceid,Title,Type,Issn,Publisher,Open Access,Open Access Diamond,SJR,SJR Best Quartile,...,Ref. / Doc.,%Female,Overton,SDG,Country,Region,Publisher.1,Coverage,Categories,Areas
0,1,28773,Ca-A Cancer Journal for Clinicians,journal,"15424863, 00079235",John Wiley and Sons Inc,No,No,"145,004",Q1,...,"62,88","48,21",4,37,United States,Northern America,John Wiley and Sons Inc,1950-2025,Hematology (Q1); Oncology (Q1),Medicine
1,2,19434,MMWR Recommendations and Reports,journal,"10575987, 15458601",Centers for Disease Control and Prevention (CDC),Yes,No,"41,754",Q1,...,"275,33","75,93",1,5,United States,Northern America,Centers for Disease Control and Prevention (CDC),1990-2024,Epidemiology (Q1); Health Information Manageme...,Environmental Science; Health Professions; Med...
2,3,20315,Nature Reviews Molecular Cell Biology,journal,"14710072, 14710080",Nature Research,No,No,"37,353",Q1,...,"92,45","41,22",0,15,United Kingdom,Western Europe,Nature Research,2000-2025,Cell Biology (Q1); Molecular Biology (Q1),"Biochemistry, Genetics and Molecular Biology"
3,4,29431,Quarterly Journal of Economics,journal,"00335533, 15314650",Oxford University Press,No,No,"35,995",Q1,...,"69,79","25,18",35,27,United Kingdom,Western Europe,Oxford University Press,1886-2025,Economics and Econometrics (Q1),"Economics, Econometrics and Finance"
4,5,20425,Nature Reviews Drug Discovery,journal,"14741784, 14741776",Nature Research,No,No,"30,506",Q1,...,"35,66","26,67",1,58,United Kingdom,Western Europe,Nature Research,2002-2025,Drug Discovery (Q1); Medicine (miscellaneous) ...,"Medicine; Pharmacology, Toxicology and Pharmac..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31131,31132,145515,Waves in Random and Complex Media (discontinued),journal,"17455049, 17455030",Taylor and Francis Ltd.,No,No,NaN,-,...,"49,05","25,14",0,20,United Kingdom,Western Europe,Taylor and Francis Ltd.,2005-2024,Engineering (miscellaneous); Physics and Astro...,Engineering; Physics and Astronomy
31132,31133,21101174249,Word and Music Studies,book series,15660958,Brill: Rodopi,No,No,NaN,-,...,"79,44","100,00",0,0,Netherlands,Western Europe,Brill: Rodopi,"1999-2001, 2004-2006, 2008, 2010-2011, 2014-20...",Arts and Humanities (miscellaneous); Literatur...,Arts and Humanities
31133,31134,18033,"World Dredging, Mining and Construction",trade journal,10450343,Placer Management Corp.,No,No,NaN,-,...,"0,00","100,00",0,1,United States,Northern America,Placer Management Corp.,"1979, 1988-2024",Building and Construction; Ocean Engineering; ...,Earth and Planetary Sciences; Engineering
31134,31135,21100898632,World Scientific-Now Publishers Series in Busi...,book series,22513442,World Scientific,No,No,NaN,-,...,"31,33","33,33",0,0,Singapore,Asiatic Region,World Scientific,"2018-2020, 2023-2024","Business, Management and Accounting (miscellan...","Business, Management and Accounting"


In [45]:
ql_long = ql.copy()
ql_long['Issn'] = ql_long['Issn'].str.split(',')
ql_long = ql_long.explode('Issn')
ql_long['Issn'] = ql_long['Issn'].str.strip()

# Keep SJR Best Quartile and Publisher columns
# (adjust column names if they differ in your CSV)
ql_long = ql_long[['Issn', 'SJR Best Quartile', 'Publisher']]
ql_long.head()

,Issn,SJR Best Quartile,Publisher
0,15424863,Q1,John Wiley and Sons Inc
0,00079235,Q1,John Wiley and Sons Inc
1,10575987,Q1,Centers for Disease Control and Prevention (CDC)
1,15458601,Q1,Centers for Disease Control and Prevention (CDC)
2,14710072,Q1,Nature Research


In [46]:

pubs["Issn"] = pubs["prism:issn"].fillna(pubs["prism:eIssn"])
pubs.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,prism:aggregationType,prism:url,openaccess,openaccessFlag,affiliation,year,affiliation_countries,affiliation_names,scopus_id,Issn
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,Journal,https://api.elsevier.com/content/abstract/scop...,0,False,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile; Peru,Pontificia Universidad Católica de Chile; Univ...,105015077202,09521976
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,Journal,https://api.elsevier.com/content/abstract/scop...,1,True,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile,Pontificia Universidad Católica de Chile; Univ...,105013320148,2296665X
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,Journal,https://api.elsevier.com/content/abstract/scop...,2,False,"[{'@_fa': 'true', 'affilname': 'RMGEO Consulta...",2024,Canada,RMGEO Consultants Inc.,85185156225,09218009


In [47]:
pubs_q = pubs.merge(ql_long, how="left", left_on=["Issn"], right_on=["Issn"])

pubs_q.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,openaccess,openaccessFlag,affiliation,year,affiliation_countries,affiliation_names,scopus_id,Issn,SJR Best Quartile,Publisher
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,0,False,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile; Peru,Pontificia Universidad Católica de Chile; Univ...,105015077202,09521976,Q1,Elsevier Ltd
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,1,True,"[{'@_fa': 'true', 'affilname': 'Pontificia Uni...",2025,Chile,Pontificia Universidad Católica de Chile; Univ...,105013320148,2296665X,Q1,Frontiers Media SA
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,2,False,"[{'@_fa': 'true', 'affilname': 'RMGEO Consulta...",2024,Canada,RMGEO Consultants Inc.,85185156225,09218009,Q1,Elsevier B.V.


In [48]:
pubs_q.to_csv(os.path.join(OUT_DIR, "scopus_publications.csv"), index=False)
print("Quartile join placeholder ready.")

Quartile join placeholder ready.


## 9) Sanity checks

In [49]:
# Basic checks
assert pubs["author_id"].notna().all()
print("Unique professors:", pubs["prof_id"].nunique())
print("Years covered:", pubs["year"].min(), "-", pubs["year"].max())
print("Total publications:", len(pubs))

# Example: publications per author
pubs.groupby(["prof_id","prof_name"]).size().sort_values(ascending=False).head(10)


Unique professors: 20
Years covered: 2019 - 2025
Total publications: 167


prof_id  prof_name                
TEC-019  Silverio-Murillo, Adan       26
TEC-001  Molina-Perez, Edmundo        15
TEC-003  Bernal-Serrano, Daniel       15
TEC-018  Diaz-Dominguez, Alejandro    13
TEC-015  Ponce-Lopez, Roberto         13
TEC-016  Gomez-Zaldivar, Fernando     10
TEC-004  Aguilar-Gomez, Sandra        10
TEC-009  Popper, Steven W.             9
TEC-014  Contreras-Loya, David         9
TEC-010  Peraza-Mues, Gonzalo G.       7
dtype: int64